# GAT ogbl-collab Evaluation

This notebook reads the JSON logs produced by `scripts/train.py` and summarizes loss curves and Hits@K results.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

LOG_CANDIDATES = [
    Path('../logs/rtx5070ti/training_log.json'),
    Path('../logs/training_log.json'),
]
LOG_PATH = next((path for path in LOG_CANDIDATES if path.exists()), LOG_CANDIDATES[0])

if not LOG_PATH.exists():
    raise FileNotFoundError('Run `python scripts/train.py` before opening this notebook.')

log = json.loads(LOG_PATH.read_text())
len(log['records']), [len(run) for run in log['records']]

## Loss Curves

In [ ]:
plt.figure(figsize=(10, 4))
for run, losses in enumerate(log['losses'], start=1):
    if losses:
        plt.plot(range(1, len(losses) + 1), losses, label=f'run {run}')
plt.xlabel('Epoch')
plt.ylabel('Training loss')
plt.title('Training loss')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## Best Results

In [ ]:
rows = []
for metric, best_by_run in log['best'].items():
    for run, item in enumerate(best_by_run, start=1):
        rows.append({
            'metric': metric,
            'run': run,
            'best_valid_percent': 100 * item['best_valid'],
            'test_at_best_valid_percent': 100 * item['test_at_best_valid'],
            'eval_index': item['epoch_index'],
        })

summary = pd.DataFrame(rows)
summary

## Metric Curves

In [ ]:
records = []
for run, run_records in enumerate(log['records'], start=1):
    for record in run_records:
        for metric, values in record['metrics'].items():
            for split, value in values.items():
                records.append({
                    'run': run,
                    'epoch': record['epoch'],
                    'metric': metric,
                    'split': split,
                    'value_percent': 100 * value,
                })

curves = pd.DataFrame(records)
for metric in sorted(curves['metric'].unique()):
    fig, ax = plt.subplots(figsize=(10, 4))
    subset = curves[curves['metric'] == metric]
    for (run, split), frame in subset.groupby(['run', 'split']):
        ax.plot(frame['epoch'], frame['value_percent'], label=f'run {run} {split}')
    ax.set_title(metric)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Hits (%)')
    ax.grid(alpha=0.25)
    ax.legend(ncol=3)
    plt.show()